# Recording-Site Impedance and Noise Model for a Penetrating Neural Probe

Models the electrode-tissue interface of a microelectrode recording site as a spreading resistance in series with a
constant-phase element (CPE) in parallel with a charge-transfer resistance, then estimates thermal noise and the signal
loss caused by the amplifier input capacitance.

**How to use:** Runtime > Run all. Change the inputs in the "Inputs" cell and run again.

**Important:** the coating parameters below are illustrative placeholders. Replace them with values fitted from the
literature (or your own EIS data) before using any number in a paper.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

K_B = 1.380649e-23   # Boltzmann constant, J/K
T_BODY = 310.0       # temperature, K
_trapz = getattr(np, "trapezoid", None) or np.trapz

## Inputs

In [ ]:
SITE_DIAMETERS_UM = [10, 15, 20, 30, 50]     # recording-site diameters (um)
COATINGS = ["Pt", "TiN", "IrOx", "PEDOT"]
REF_DIAMETER_UM = 20                          # diameter used in the Bode plot

F_MIN, F_MAX = 1.0, 1e5                       # plot range, Hz
BAND = (300.0, 5000.0)                        # spike band for RMS noise, Hz

RHO_OHM_M = 0.7                               # medium resistivity (saline/CSF ~0.7, brain tissue ~2-4)
AMP_NOISE_NV_RTHZ = 5.0                       # amplifier input-referred voltage noise, nV/sqrt(Hz)
AMP_CIN_PF = 5.0                              # amplifier input + stray capacitance, pF
SPIKE_AMPLITUDE_UV = 100.0                    # example extracellular spike amplitude, uV

# CPE parameters per unit area (PLACEHOLDERS - replace with literature or measured values)
# Q0: F*s^(n-1) per cm^2, n: CPE exponent, Rct: ohm*cm^2
COATING_PARAMS = {
    "Pt":    dict(Q0=20e-6,   n=0.90, Rct=1e6),
    "TiN":   dict(Q0=300e-6,  n=0.90, Rct=1e5),
    "IrOx":  dict(Q0=800e-6,  n=0.85, Rct=1e5),
    "PEDOT": dict(Q0=2000e-6, n=0.85, Rct=1e5),
}

## Model

In [ ]:
def site_area_cm2(d_um):
    r_cm = d_um * 1e-4 / 2.0
    return np.pi * r_cm ** 2

def spreading_resistance(d_um, rho=RHO_OHM_M):
    # disc electrode in a semi-infinite medium: Rs = rho / (4a)
    a = d_um * 1e-6 / 2.0
    return rho / (4.0 * a)

def impedance(f, d_um, coating, rho=RHO_OHM_M):
    p = COATING_PARAMS[coating]
    A = site_area_cm2(d_um)
    w = 2 * np.pi * np.asarray(f, dtype=float)
    z_cpe = 1.0 / ((p["Q0"] * A) * (1j * w) ** p["n"])
    r_ct = p["Rct"] / A
    z_int = 1.0 / (1.0 / r_ct + 1.0 / z_cpe)
    return spreading_resistance(d_um, rho) + z_int

def amp_transfer(f, z_site, c_in_pf=AMP_CIN_PF):
    # voltage divider between site impedance and amplifier input capacitance
    w = 2 * np.pi * np.asarray(f, dtype=float)
    z_in = 1.0 / (1j * w * c_in_pf * 1e-12)
    return z_in / (z_in + z_site)

def rms_noise_uv(d_um, coating, band=BAND, rho=RHO_OHM_M):
    f = np.logspace(np.log10(band[0]), np.log10(band[1]), 500)
    z = impedance(f, d_um, coating, rho)
    h = np.abs(amp_transfer(f, z))
    psd_electrode = 4 * K_B * T_BODY * np.real(z) * h ** 2     # V^2/Hz at amplifier input
    psd_amp = (AMP_NOISE_NV_RTHZ * 1e-9) ** 2                   # V^2/Hz
    total = np.sqrt(_trapz(psd_electrode + psd_amp, f))
    electrode_only = np.sqrt(_trapz(psd_electrode, f))
    gain = float(np.mean(h))
    return total * 1e6, electrode_only * 1e6, gain

## Bode plot (reference diameter, all coatings)

In [ ]:
f = np.logspace(np.log10(F_MIN), np.log10(F_MAX), 400)
fig, ax = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
for c in COATINGS:
    z = impedance(f, REF_DIAMETER_UM, c)
    ax[0].loglog(f, np.abs(z), label=c)
    ax[1].semilogx(f, np.degrees(np.angle(z)), label=c)
ax[0].set_ylabel("|Z| (ohm)")
ax[1].set_ylabel("Phase (deg)")
ax[1].set_xlabel("Frequency (Hz)")
ax[0].set_title(f"Recording site, diameter {REF_DIAMETER_UM} um")
ax[0].legend(); ax[0].grid(True, which="both", alpha=0.3); ax[1].grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("bode.png", dpi=200)
plt.show()

## Nyquist plot (reference diameter)

In [ ]:
plt.figure(figsize=(6, 5))
for c in COATINGS:
    z = impedance(f, REF_DIAMETER_UM, c)
    plt.plot(np.real(z) / 1e3, -np.imag(z) / 1e3, label=c)
plt.xlabel("Re(Z) (kohm)"); plt.ylabel("-Im(Z) (kohm)")
plt.title(f"Nyquist, diameter {REF_DIAMETER_UM} um")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("nyquist.png", dpi=200)
plt.show()

## Sweep: site diameter x coating

In [ ]:
rows = []
for c in COATINGS:
    for d in SITE_DIAMETERS_UM:
        z1k = impedance(1000.0, d, c)
        total, elec, gain = rms_noise_uv(d, c)
        snr = (SPIKE_AMPLITUDE_UV * gain) / total
        rows.append({
            "coating": c,
            "diameter_um": d,
            "|Z| @1kHz (kohm)": round(float(np.abs(z1k)) / 1e3, 2),
            "electrode noise (uVrms)": round(elec, 3),
            "total noise (uVrms)": round(total, 3),
            "mean signal gain": round(gain, 3),
            "spike SNR": round(snr, 2),
        })
df = pd.DataFrame(rows)
df.to_csv("impedance_noise_sweep.csv", index=False)
df

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for c in COATINGS:
    sub = df[df.coating == c]
    ax[0].loglog(sub.diameter_um, sub["|Z| @1kHz (kohm)"], "o-", label=c)
    ax[1].semilogx(sub.diameter_um, sub["total noise (uVrms)"], "o-", label=c)
    ax[2].semilogx(sub.diameter_um, sub["spike SNR"], "o-", label=c)
ax[0].set_ylabel("|Z| @ 1 kHz (kohm)")
ax[1].set_ylabel("Total noise (uVrms, 300 Hz - 5 kHz)")
ax[2].set_ylabel(f"Spike SNR ({SPIKE_AMPLITUDE_UV:.0f} uV spike)")
for a in ax:
    a.set_xlabel("Site diameter (um)"); a.grid(True, which="both", alpha=0.3); a.legend()
plt.tight_layout()
plt.savefig("sweep.png", dpi=200)
plt.show()

## Next steps for the paper
1. Replace the placeholder CPE parameters with values fitted from published EIS data for each coating.
2. Add the shank geometry and buckling-force calculation (next notebook).
3. Compare the model against reported impedance values for known probes (e.g. Michigan-style and Neuropixels sites).